# REINFORCE on a Grid World

This notebook is a beginner-friendly walkthrough of the REINFORCE algorithm.

You will:

- build a small grid world with a start, a goal, and obstacles;
- use a stochastic policy, which means the agent chooses actions using probabilities;
- sample episodes and compute discounted returns;
- update the policy so actions that led to better returns become more likely;
- visualize training curves, policy arrows, and rollout paths.

The default policy is deliberately tabular: each grid cell has four learnable action scores, one for each direction. That keeps the mechanics visible before moving to neural-network policies.


## Setup

Run the notebook from top to bottom. If you change an earlier setup cell, rerun the cells below it so they use the new settings.

The first code cell does four things:

1. clones this repo automatically if you are running in Google Colab;
2. sets a Matplotlib cache folder so plotting works cleanly;
3. imports PyTorch, plotting tools, and helper functions;
4. fixes the random seed so repeated runs are easier to compare.

Most of the longer helper code lives in the `reinforce_demo/` Python files next to this notebook. You do not need to read those files first. If you do open them, each file has a short purpose:

- `environment.py`: grid-world rules and rewards;
- `policies.py`: policy classes, episode collection, and the REINFORCE update;
- `training.py`: the repeated training/evaluation loops;
- `plotting.py`: figures and animations.

If the imports fail, install the dependencies from a terminal with:

```bash
pip install -r requirements.txt
```

The notebook uses CPU PyTorch only; no GPU is required.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not (Path.cwd() / 'reinforce_demo').exists():
    repo_dir = Path('/content/RL-Demo')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Paul-Lez/RL-Demo.git', str(repo_dir)], check=True)
    os.chdir(repo_dir)

sys.path.insert(0, os.getcwd())

# Matplotlib writes a small cache. Keeping it in this folder avoids permission
# issues in some notebook/Docker setups.
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.matplotlib'))
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)

try:
    import torch
except ImportError as exc:
    raise ImportError(
        'PyTorch is required for this notebook. Install dependencies with: pip install -r requirements.txt'
    ) from exc

import matplotlib.pyplot as plt

# Environment rules and reward descriptions.
from reinforce_demo.environment import GridWorld, REWARD_MODES

# Policy classes and the core REINFORCE pieces.
from reinforce_demo.policies import (
    POLICY_MODES,
    UPDATE_MODES,
    collect_episode,
    default_learning_rate,
    make_policy,
    reinforce_update,
    resolve_policy_mode,
    resolve_update_mode,
    set_seed,
)

# Plotting helpers used to make the learning process visible.
from reinforce_demo.plotting import (
    animate_episode,
    configure_plots,
    plot_episode_credit,
    plot_grid,
    plot_policy,
    plot_policy_from_probs,
    plot_policy_snapshots,
    plot_probability_deltas,
    plot_sampled_trajectories,
    plot_training,
    policy_action_probs,
    summarize_episode_update,
)
from reinforce_demo.training import evaluate_policy, sample_policy_trajectories, train_reinforce

set_seed(7)
configure_plots()


## The REINFORCE Idea

The policy is stochastic. At each state it returns probabilities over the four actions:

`up`, `right`, `down`, `left`.

A single episode is one complete attempt: start at `S`, move around the grid, and stop when the agent reaches `G` or runs out of time.

For each time step, REINFORCE asks: after this action, how much reward did the agent eventually collect? That future total is called the **return** and is usually written as `G_t`.

Vanilla REINFORCE uses the raw return directly:

```python
loss = -sum(log_prob_t * return_t)
```

This notebook also has an `advantage` mode, which keeps the same sampled-policy idea but uses a normalized return-as-advantage signal and an optional entropy bonus for exploration.

Here `log_prob_t` tells PyTorch which sampled action to adjust, and the return or advantage signal says whether that action worked better or worse than usual. Positive signals push the sampled action up; negative signals push it down.

You do not need to memorize the equation to follow the demo. The important idea is: **sample an episode, score the actions by what happened afterward, then adjust the policy probabilities.**


## Environment

The next code cell creates the world the agent will learn in.

The agent starts at `S` in the lower-left corner and tries to reach `G` in the upper-right corner. Dark cells are obstacles. A state is written as `(row, column)`, so `(0, 0)` is the top-left cell.

You can change `N`, `obstacles`, and `REWARD_MODE` below. Start with the default values first, then experiment after you have seen the full notebook once.

Reward modes:

- `dense`: small step cost, small blocked-move penalty, and a goal reward;
- `sparse`: zero reward until the goal, then a goal reward;
- `sparse_length`: zero reward until the goal, then a smaller reward for longer paths;
- `dense_noop_penalty`: like `dense`, but actions that hit walls or obstacles are penalized more strongly.


In [ ]:
# Grid size. The grid has N rows and N columns.
N = 6

# Start with the easiest reward signal. Later, try 'sparse' or 'sparse_length'.
REWARD_MODE = 'dense'

# Obstacles are blocked cells written as (row, column).
obstacles = {
    (1, 1),
    (2, 3),
    (3, 1),
    (4, 4),
}

env = GridWorld(n=N, obstacles=obstacles, reward_mode=REWARD_MODE)
print(f'Reward mode: {env.reward_mode} - {REWARD_MODES[env.reward_mode]}')
plot_grid(env, title='Grid world: start S, goal G, dark cells are obstacles');


## Policy and Episode Collection

A policy turns a state into action probabilities. We will start with `tabular`, which is the easiest version to inspect.

In the plot, each arrow shows the probability of choosing that direction from that cell. At the start, the tabular policy has equal action scores, so the arrows should look roughly balanced.

Policy modes:

- `tabular`: one learnable vector of four action scores per grid cell;
- `mlp`: a small neural network that maps a one-hot state representation to action scores;
- `large_mlp`: a larger neural network, useful later for seeing how neural RL can be sample-inefficient.

The training loop works the same way for all three modes.


In [ ]:
# Use 'tabular' first because it is easiest to interpret cell by cell.
POLICY_MODE = 'tabular'
POLICY_MODE = resolve_policy_mode(POLICY_MODE)
print(f'Policy mode: {POLICY_MODE} - {POLICY_MODES[POLICY_MODE]}')

# Use 'vanilla' for raw REINFORCE, or 'advantage' for the stabilized version.
UPDATE_MODE = 'advantage'
UPDATE_MODE = resolve_update_mode(UPDATE_MODE)
print(f'Update mode: {UPDATE_MODE} - {UPDATE_MODES[UPDATE_MODE]}')

# This policy has not learned yet, so it should look close to uniform.
initial_policy = make_policy(env.num_states, env.num_actions, policy_mode=POLICY_MODE)
plot_policy(env, initial_policy, title=f'Initial {POLICY_MODE} policy');


## What Does One Policy Update Do?

Before training for many episodes, inspect one sampled episode and one update.

The code below makes a fresh policy, samples one episode, prints the learning signal for that episode, then applies exactly one REINFORCE update.

The table shows:

- the state the agent was in;
- the action it sampled;
- the immediate reward;
- the discounted return from that step onward;
- the learning signal used by the gradient step.

After the update, the final heatmaps show which action probabilities went up or down. This is the core REINFORCE idea in miniature: sampled actions followed by good outcomes get reinforced, and sampled actions followed by poor outcomes get discouraged.


In [ ]:
# Use a separate policy so this one-update demo does not affect later training.
set_seed(3)
probe_policy = make_policy(env.num_states, env.num_actions, policy_mode=POLICY_MODE)
probe_lr = {'tabular': 0.08, 'mlp': 0.02, 'large_mlp': 0.005}[POLICY_MODE]
probe_optimizer = torch.optim.Adam(probe_policy.parameters(), lr=probe_lr)

before = policy_action_probs(probe_policy, env)

# Sample one episode and print the quantities REINFORCE uses to learn from it.
episode = collect_episode(env, probe_policy, greedy=False, track_grad=True)
summarize_episode_update(env, episode, gamma=0.98, update_mode=UPDATE_MODE)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
plot_policy_from_probs(env, before, ax=axes[0], title='Before one update')
plot_episode_credit(env, episode, gamma=0.98, ax=axes[1])
plt.show()

# Apply exactly one policy-gradient update, then compare probabilities.
_ = reinforce_update(
    probe_policy,
    probe_optimizer,
    episode,
    gamma=0.98,
    entropy_coef=0.0,
    update_mode=UPDATE_MODE,
)
after = policy_action_probs(probe_policy, env)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
plot_policy_from_probs(env, before, ax=axes[0], title='Before')
plot_policy_from_probs(env, after, ax=axes[1], title='After one REINFORCE update')
plt.show()

plot_probability_deltas(env, before, after);


## Training Loop

Now train for many episodes. Each episode follows the same pattern:

1. sample a trajectory from the current policy;
2. compute discounted returns and the selected learning signal;
3. apply the REINFORCE update;
4. record learning curves and occasional policy snapshots.

Important settings in the code cell:

- `episodes=900`: how many attempts the agent gets;
- `UPDATE_MODE`: `vanilla` uses raw returns; `advantage` uses the stabilized signal;
- `gamma=0.98`: how strongly future rewards count;
- `entropy_coef=0.01`: a small bonus for keeping the policy exploratory in `advantage` mode;
- `snapshot_every=150`: how often to save policy arrows for later plots.

The grey training curves are noisy because each episode is sampled. The darker curves show a moving average, which makes the trend easier to see.


In [ ]:
set_seed(7)
policy = make_policy(env.num_states, env.num_actions, policy_mode=POLICY_MODE)
learning_rate = default_learning_rate(POLICY_MODE)
optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)
print(f'Training {POLICY_MODE} policy with {UPDATE_MODE} update mode and lr={learning_rate}')

stats, snapshots = train_reinforce(
    env,
    policy,
    optimizer,
    episodes=900,
    gamma=0.98,
    entropy_coef=0.01,
    update_mode=UPDATE_MODE,
    snapshot_every=150,
    live=False,
)

# `stats` holds learning curves. `snapshots` holds saved policy arrows.
plot_training(stats, window=50);


## What Changed During Optimization?

The snapshots below show how the policy changes during training. At the start, most arrows are similar because the policy is nearly uniform. Later, larger arrows should point toward routes that reach the goal more reliably.

The second plot compares the initial and trained policies directly, then shows probability changes for each action. Red and blue do not mean good or bad here; they mean that a probability went up or down.

The policy still represents probabilities, not a hard-coded path. Even after training, it can sometimes sample a different action.


In [ ]:
plot_policy_snapshots(env, snapshots, episodes_to_show=[0, 150, 300, 600, 900]);


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
plot_policy_from_probs(env, snapshots[0], ax=axes[0], title='initial policy')
plot_policy(env, policy, ax=axes[1], title='trained policy')
plt.show()

plot_probability_deltas(env, snapshots[0], policy_action_probs(policy, env));


## Evaluate and Animate a Rollout

For evaluation, we can use the greedy action at each state: choose the action with the highest policy probability.

The first cell reports average return, episode length, and success rate. The next cells draw and animate one greedy rollout.

This gives a clean summary of what the policy currently prefers. It is different from sampling, because sampling still allows lower-probability actions sometimes.


In [ ]:
# Greedy evaluation chooses the highest-probability action at every state.
evaluate_policy(env, policy, episodes=100, greedy=True)


In [ ]:
# Collect one greedy episode so we can draw the path the policy prefers most.
greedy_episode = collect_episode(env, policy, greedy=True, track_grad=False)
trajectory = greedy_episode['states'] + greedy_episode['next_states'][-1:]
plot_grid(env, title='Greedy trajectory after training', trajectory=trajectory);


In [ ]:
animate_episode(env, greedy_episode)


## Sample Several Stochastic Rollouts

The greedy rollout shows the most likely action at each state. To see the learned stochastic policy itself, sample several rollouts from the policy distribution.

Each printed line reports the total reward, number of steps, and whether that sampled rollout reached the goal.

This is useful because REINFORCE learns probabilities. A trained policy can strongly prefer a good path while still occasionally exploring other actions.


In [ ]:
NUM_SAMPLED_TRAJECTORIES = 5
sampled_episodes = sample_policy_trajectories(env, policy, num_trajectories=NUM_SAMPLED_TRAJECTORIES, seed=21)

for i, episode in enumerate(sampled_episodes, start=1):
    print(
        f'rollout {i}: return={sum(episode["rewards"]):.3f}, '
        f'length={len(episode["rewards"])}, '
        f'success={episode["infos"][-1].reached_goal}'
    )

plot_sampled_trajectories(env, sampled_episodes);


## Things To Try

- Change `REWARD_MODE` from `dense` to `sparse`. Does learning become harder when rewards are delayed?
- Change `UPDATE_MODE` from `advantage` to `vanilla`. How much noisier is raw REINFORCE?
- Change `POLICY_MODE` from `tabular` to `mlp`. Does the neural policy learn the same behavior?
- Try `POLICY_MODE = 'large_mlp'`. Does the extra capacity help here, or does it make optimization slower?
- Increase `N` or add more obstacles. How does the amount of exploration change?
- Set `entropy_coef=0.0` in the training cell while using `advantage`. Does the policy stop exploring too early?
- Change `gamma`. With a smaller discount, does the agent still care enough about the distant goal?
- Set `live=True` in `train_reinforce` if you want the training cell to refresh while it runs.
- Try tweaking the rewards a little: can you find one that fully eliminates no-op moves?


## More Fun: 
- Try changing the environment, e.g. make it larger (i.e. change `N`), add traps (i.e. squares that result in a failure when entered). How does this change the learning process?
- Try implementing another RL algorithm for this setting!
